# COMS 6998 — High Performance Machine Learning
## Homework 4: Quantization & Pruning
**Name:** Rajvardhan Patil  
**UNI:** rp3359  
**Due:** April 20, Spring 2026

## Setup

First things first — imports, data loading, and training a baseline model. I'm using a simple 2-conv CNN on CIFAR-10. Nothing fancy, just enough to get a decent accuracy to work with.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.utils.prune as prune
import matplotlib.pyplot as plt
import numpy as np
import copy
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# standard CIFAR-10 normalization
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=128, shuffle=False, num_workers=2)

# reuse trainloader for PTQ calibration in Q4
calib_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=False, num_workers=2)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print('data loaded.')

In [ ]:
class SimpleCNN(nn.Module):
    # 2 conv layers -> 2 fc layers, pretty standard for CIFAR-10
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 512)
        self.fc2   = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 -> 8x8
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


def train_model(model, loader, optimizer, criterion, epochs=10):
    model.train()
    loss_hist = []
    for epoch in range(epochs):
        total_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(inputs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * inputs.size(0)
        avg = total_loss / len(loader.dataset)
        loss_hist.append(avg)
        print(f'  epoch [{epoch+1}/{epochs}]  loss: {avg:.4f}')
    return loss_hist


def evaluate(model, loader, dev=device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(dev), labels.to(dev)
            out = model(inputs)
            _, pred = out.max(1)
            total   += labels.size(0)
            correct += pred.eq(labels).sum().item()
    return 100.0 * correct / total


# train the baseline
baseline_model = SimpleCNN().to(device)
criterion      = nn.CrossEntropyLoss()
optimizer      = optim.SGD(baseline_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler      = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 8], gamma=0.1)

print('training baseline ...')
loss_hist = train_model(baseline_model, trainloader, optimizer, criterion, epochs=10)
for _ in range(10):
    scheduler.step()

baseline_acc = evaluate(baseline_model, testloader)
print(f'\nbaseline test accuracy: {baseline_acc:.2f}%')

torch.save(baseline_model.state_dict(), 'baseline_cnn.pth')

---
## Q1: Visualize Weights (15 pts)

I want to see what the weight distributions actually look like for each layer. The idea is that before we quantize anything, we should understand the range and shape of the values we're working with. I'm just pulling out the weight tensors and plotting histograms — nothing complicated here.

In [ ]:
named_layers = [
    ('conv1', baseline_model.conv1),
    ('conv2', baseline_model.conv2),
    ('fc1',   baseline_model.fc1),
    ('fc2',   baseline_model.fc2),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Q1 — Weight Distributions', fontsize=13)

for ax, (name, layer) in zip(axes, named_layers):
    w = layer.weight.data.cpu().numpy().flatten()
    ax.hist(w, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'{name}  ({len(w):,} params)')
    ax.set_xlabel('weight value')
    ax.set_ylabel('count')
    mu = w.mean()
    ax.axvline(mu, color='red', linestyle='--', lw=1.5, label=f'mean={mu:.3f}')
    ax.legend(fontsize=8)
    print(f'{name}: min={w.min():.4f}  max={w.max():.4f}  mean={mu:.4f}  std={w.std():.4f}')

plt.tight_layout()
plt.savefig('q1_weight_distributions.png', dpi=150)
plt.show()

---
## Q2: Quantize Weights (15 pts)

Here I'm implementing 8-bit symmetric quantization by hand. The math is pretty straightforward:

- compute a scale factor: `scale = max(|W|) / 127`
- map each float weight to the nearest int8: `W_q = clamp(round(W / scale), -128, 127)`
- dequantize back to float: `W_deq = W_q * scale`

The dequantized weights won't match the originals exactly — that rounding error is basically the quantization noise. I'll plot both histograms side by side to see how much the distribution changes, and also compute the MSE per layer.

In [ ]:
def quantize_weights_int8(tensor):
    scale = tensor.abs().max() / 127.0
    if scale == 0:
        return tensor.clone().to(torch.int8), tensor.clone(), scale
    q   = torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
    deq = q.to(torch.float32) * scale
    return q, deq, scale


# deep copy so baseline stays intact
q2_model = copy.deepcopy(baseline_model)

fig, axes = plt.subplots(4, 2, figsize=(12, 16))
fig.suptitle('Q2 — Original vs Dequantized Weights', fontsize=13)

mse_results = {}
for i, (name, layer) in enumerate(named_layers):
    W_orig = layer.weight.data.cpu()
    _, deq_w, scale = quantize_weights_int8(W_orig)

    # swap in the dequantized weights
    target = getattr(q2_model, name)
    with torch.no_grad():
        target.weight.copy_(deq_w)

    mse = ((W_orig - deq_w) ** 2).mean().item()
    mse_results[name] = mse

    axes[i][0].hist(W_orig.numpy().flatten(), bins=60, color='steelblue', alpha=0.8, edgecolor='none')
    axes[i][0].set_title(f'{name} — original (FP32)')
    axes[i][0].set_xlabel('weight value')

    axes[i][1].hist(deq_w.numpy().flatten(), bins=60, color='darkorange', alpha=0.8, edgecolor='none')
    axes[i][1].set_title(f'{name} — dequantized  scale={scale:.5f}')
    axes[i][1].set_xlabel('weight value')

    print(f'{name}: scale={scale:.6f}   MSE={mse:.2e}')

plt.tight_layout()
plt.savefig('q2_quantized_weights.png', dpi=150)
plt.show()

q2_acc = evaluate(q2_model.to(device), testloader)
print(f'\nbaseline:          {baseline_acc:.2f}%')
print(f'weight-quantized:  {q2_acc:.2f}%')
print(f'drop:              {baseline_acc - q2_acc:.2f}%')

---
## Q3: Visualize Activations (15 pts)

Now I want to look at activation distributions — what values are actually flowing through the network during inference. I'll register forward hooks on each layer to grab the outputs, run a few batches through the model, then plot histograms.

One thing to notice: after ReLU, conv layer activations are all non-negative (nothing below zero). The FC layers don't have ReLU on the output side so they can be negative too.

In [ ]:
act_store = {}  # layer name -> list of output tensors
hooks = []

def make_hook(name):
    def hook(module, inp, out):
        # grab a CPU copy so we don't hog GPU memory
        act_store.setdefault(name, []).append(out.detach().cpu())
    return hook

hook_targets = {
    'conv1_relu': baseline_model.conv1,
    'conv2_relu': baseline_model.conv2,
    'fc1':        baseline_model.fc1,
    'fc2':        baseline_model.fc2,
}
for name, mod in hook_targets.items():
    hooks.append(mod.register_forward_hook(make_hook(name)))

baseline_model.eval()
with torch.no_grad():
    for idx, (inputs, _) in enumerate(testloader):
        _ = baseline_model(inputs.to(device))
        if idx >= 4:  # ~640 images, enough for a reasonable histogram
            break

for h in hooks:
    h.remove()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Q3 — Activation Distributions', fontsize=13)

for ax, (name, tensors) in zip(axes, act_store.items()):
    flat = torch.cat([t.flatten() for t in tensors]).numpy()
    ax.hist(flat, bins=60, color='mediumseagreen', edgecolor='none', alpha=0.85)
    ax.set_title(f'{name}')
    ax.set_xlabel('activation value')
    ax.set_ylabel('count')
    print(f'{name}: min={flat.min():.4f}  max={flat.max():.4f}  mean={flat.mean():.4f}  std={flat.std():.4f}')

plt.tight_layout()
plt.savefig('q3_activation_distributions.png', dpi=150)
plt.show()

---
## Q4: Quantize Activations — Static PTQ (15 pts)

This is where PyTorch's built-in quantization API comes in. Static PTQ works like this:

1. Wrap the model with `QuantStub` / `DeQuantStub` at the input/output boundaries
2. Set a qconfig — I'm using `fbgemm` which targets x86 CPUs
3. Call `prepare()` to insert observer modules that track activation ranges
4. Run some training data through the prepared model (calibration step — observers collect stats)
5. Call `convert()` to fold everything into actual INT8 ops

One gotcha: `fbgemm` is CPU-only, so I had to move everything off the GPU for this part.

In [ ]:
# fbgemm is CPU-only, so this whole section runs on CPU

class QuantizableCNN(nn.Module):
    # same architecture as SimpleCNN, but with quant/dequant stubs at the boundaries
    def __init__(self, base):
        super().__init__()
        self.quant   = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        self.conv1 = copy.deepcopy(base.conv1)
        self.conv2 = copy.deepcopy(base.conv2)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = copy.deepcopy(base.fc1)
        self.fc2   = copy.deepcopy(base.fc2)

    def forward(self, x):
        x = self.quant(x)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)
        return x


q4_model = QuantizableCNN(baseline_model).cpu().eval()
q4_model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(q4_model, inplace=True)

# calibration — observers watch activations over ~1k images
print('calibrating ...')
with torch.no_grad():
    for idx, (inputs, _) in enumerate(calib_loader):
        q4_model(inputs.cpu())
        if idx >= 7:
            break
print('done.')

torch.quantization.convert(q4_model, inplace=True)
print('converted to INT8.')


def evaluate_cpu(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            out = model(inputs.cpu())
            _, pred = out.max(1)
            total   += labels.size(0)
            correct += pred.eq(labels.cpu()).sum().item()
    return 100.0 * correct / total


q4_acc = evaluate_cpu(q4_model, testloader)
print(f'\nbaseline:       {baseline_acc:.2f}%')
print(f'static PTQ:     {q4_acc:.2f}%')
print(f'drop:           {baseline_acc - q4_acc:.2f}%')

# model size comparison
torch.save(baseline_model.state_dict(), '/tmp/fp32_model.pth')
torch.save(q4_model.state_dict(),       '/tmp/int8_model.pth')

fp32_kb = os.path.getsize('/tmp/fp32_model.pth') / 1024
int8_kb = os.path.getsize('/tmp/int8_model.pth') / 1024
print(f'\nFP32 size: {fp32_kb:.1f} KB')
print(f'INT8 size: {int8_kb:.1f} KB')
print(f'compression: {fp32_kb / int8_kb:.2f}x')

---
## Q5: Quantize Biases (10 pts)

Biases get quantized differently from weights — the standard approach is INT32 rather than INT8. The reason is that the bias scale depends on both the weight scale AND the input activation scale, which makes it much smaller. Squeezing that into 8 bits would introduce way too much rounding error.

The formula I'm using:
- `s_b = s_w * s_x`  (combined scale)
- `b_q = clamp(round(b / s_b), -2^31, 2^31-1)`

I pull the activation scales from the Q3 hook data (just `max(|activations|) / 127`).

In [ ]:
def quantize_bias_int32(bias, scale_w, scale_x):
    scale_b = scale_w * scale_x
    INT32_MAX = 2**31 - 1
    b_q   = torch.clamp(torch.round(bias / scale_b), -INT32_MAX - 1, INT32_MAX).to(torch.int32)
    b_deq = b_q.to(torch.float32) * scale_b
    return b_q, b_deq, scale_b


# pull activation scales from Q3 hook data
act_scales = {}
for name, tensors in act_store.items():
    flat = torch.cat([t.flatten() for t in tensors])
    act_scales[name] = float(flat.abs().max()) / 127.0

layer_act_key = {
    'conv1': 'conv1_relu',
    'conv2': 'conv2_relu',
    'fc1':   'fc1',
    'fc2':   'fc2',
}

q5_model = copy.deepcopy(baseline_model)

print(f'{"layer":<8}  {"s_w":>10}  {"s_x":>10}  {"s_b":>12}  {"bias MSE":>12}')
print('-' * 58)

for name, layer in named_layers:
    W_orig = layer.weight.data.cpu()
    b_orig = layer.bias.data.cpu()

    _, deq_w, scale_w = quantize_weights_int8(W_orig)

    scale_x = act_scales.get(layer_act_key[name], 1.0 / 127.0)
    _, deq_b, scale_b = quantize_bias_int32(b_orig, scale_w, scale_x)

    mse = ((b_orig - deq_b) ** 2).mean().item()
    print(f'{name:<8}  {scale_w:>10.6f}  {scale_x:>10.6f}  {scale_b:>12.8f}  {mse:>12.2e}')

    target = getattr(q5_model, name)
    with torch.no_grad():
        target.weight.copy_(deq_w)
        target.bias.copy_(deq_b)

q5_acc = evaluate(q5_model.to(device), testloader)
print(f'\nbaseline:              {baseline_acc:.2f}%')
print(f'weights only (Q2):     {q2_acc:.2f}%')
print(f'weights + biases (Q5): {q5_acc:.2f}%')

---
## Q6: Pruning and Fine-Tuning (10 pts)

The idea here is to zero out the smallest-magnitude weights using L1 unstructured pruning, then see how much accuracy we lose and how much we can recover with a short fine-tuning pass.

I'm testing four sparsity levels: 25%, 50%, 75%, 90%. For each one:
1. Deep-copy the baseline, prune it
2. Evaluate right away (before any recovery)
3. Fine-tune for 5 epochs at a low lr (1e-4)
4. Evaluate again

The `prune.remove()` call at the end bakes the masks into the actual weight tensors.

In [ ]:
def apply_global_pruning(model, sparsity):
    params = [
        (model.conv1, 'weight'),
        (model.conv2, 'weight'),
        (model.fc1,   'weight'),
        (model.fc2,   'weight'),
    ]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=sparsity)
    return model


def make_sparsity_permanent(model):
    # bake masks into weights so they're actually zeroed out
    for mod in [model.conv1, model.conv2, model.fc1, model.fc2]:
        prune.remove(mod, 'weight')


def actual_sparsity(model):
    total, zeros = 0, 0
    for mod in [model.conv1, model.conv2, model.fc1, model.fc2]:
        w = mod.weight.data
        total += w.numel()
        zeros += (w == 0).sum().item()
    return zeros / total


sparsity_levels = [0.25, 0.50, 0.75, 0.90]
acc_before_ft   = []
acc_after_ft    = []

for sp in sparsity_levels:
    print(f'\n--- sparsity {sp:.0%} ---')

    pruned = copy.deepcopy(baseline_model).to(device)
    apply_global_pruning(pruned, sp)
    print(f'  actual sparsity: {actual_sparsity(pruned):.2%}')

    acc_pre = evaluate(pruned, testloader)
    acc_before_ft.append(acc_pre)
    print(f'  acc before FT: {acc_pre:.2f}%')

    # fine-tune with a small lr so we don't blow up the sparse structure
    ft_opt = optim.SGD(pruned.parameters(), lr=1e-4, momentum=0.9, weight_decay=5e-4)
    pruned.train()
    for epoch in range(5):
        running = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            ft_opt.zero_grad()
            loss = criterion(pruned(inputs), labels)
            loss.backward()
            ft_opt.step()
            running += loss.item() * inputs.size(0)
        print(f'    FT epoch [{epoch+1}/5]  loss: {running/len(trainloader.dataset):.4f}')

    make_sparsity_permanent(pruned)

    acc_post = evaluate(pruned, testloader)
    acc_after_ft.append(acc_post)
    print(f'  acc after FT: {acc_post:.2f}%')

# plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([s*100 for s in sparsity_levels], acc_before_ft,
        marker='o', ls='--', color='tomato', label='before fine-tuning')
ax.plot([s*100 for s in sparsity_levels], acc_after_ft,
        marker='s', ls='-',  color='steelblue', label='after fine-tuning')
ax.axhline(baseline_acc, color='gray', ls=':', lw=1.5, label=f'baseline ({baseline_acc:.1f}%)')
ax.set_xlabel('sparsity (%)')
ax.set_ylabel('test accuracy (%)')
ax.set_title('Q6 — Pruning + Fine-Tuning')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q6_pruning_finetuning.png', dpi=150)
plt.show()

print(f'\n{"sparsity":>10}  {"before FT":>10}  {"after FT":>10}  {"gain":>8}')
print('-' * 44)
for sp, b, a in zip(sparsity_levels, acc_before_ft, acc_after_ft):
    print(f'{sp:>10.0%}  {b:>10.2f}%  {a:>10.2f}%  {a-b:>+7.2f}%')

### Q6 Reflection

**Accuracy vs sparsity:**  
At 25% sparsity the accuracy barely moves before fine-tuning — usually just 1–3% below baseline. This makes sense since magnitude pruning removes the smallest weights first, and those contribute the least anyway. The network is pretty over-parameterized for CIFAR-10, so dropping a quarter of its weights isn't a big deal.

Once you get to 75–90% sparsity the story changes. At 90% the model has lost so much capacity that accuracy can tank close to chance before any recovery step. The few remaining weights simply can't represent the function the network learned.

**Effect of fine-tuning:**  
At low/moderate sparsity (25–50%), 5 epochs of fine-tuning at lr=1e-4 basically closes the gap — you end up within ~1% of baseline. The surviving weights adjust quickly because the network still has reasonable capacity.

At 75% and especially 90% sparsity, fine-tuning helps but there's a ceiling. The sparse topology itself is the bottleneck — there just aren't enough non-zero connections to fully recover. Longer fine-tuning or a higher lr might squeeze out a bit more, but you're fighting the architecture at that point.

---
## Q7: Pruning Before vs. After Training (10 pts)

Now I want to compare two pruning strategies:
- **Post-training** (Q6): prune after the model has fully converged, then fine-tune
- **At initialization**: prune the random weights before training starts, then train from scratch for the same 10 epochs

At first I thought they might end up similar at low sparsity, but at high sparsity the difference should be pretty clear.

In [ ]:
acc_prune_at_init = []

for sp in sparsity_levels:
    print(f'\n--- pruning at init, sparsity {sp:.0%} ---')

    # start from scratch — no trained weights
    init_model = SimpleCNN().to(device)
    apply_global_pruning(init_model, sp)
    print(f'  actual sparsity: {actual_sparsity(init_model):.2%}')

    # train with the same schedule as baseline
    init_opt  = optim.SGD(init_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    init_sched = optim.lr_scheduler.MultiStepLR(init_opt, milestones=[5, 8], gamma=0.1)

    init_model.train()
    for epoch in range(10):
        running = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            init_opt.zero_grad()
            loss = criterion(init_model(inputs), labels)
            loss.backward()
            init_opt.step()
            running += loss.item() * inputs.size(0)
        init_sched.step()
        print(f'  epoch [{epoch+1}/10]  loss: {running/len(trainloader.dataset):.4f}')

    make_sparsity_permanent(init_model)

    acc = evaluate(init_model, testloader)
    acc_prune_at_init.append(acc)
    print(f'  final acc: {acc:.2f}%')

# comparison plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([s*100 for s in sparsity_levels], acc_after_ft,
        marker='s', ls='-',  color='steelblue',  label='post-training prune + fine-tune (Q6)')
ax.plot([s*100 for s in sparsity_levels], acc_prune_at_init,
        marker='^', ls='--', color='darkorange', label='prune at init + train from scratch')
ax.axhline(baseline_acc, color='gray', ls=':', lw=1.5, label=f'baseline ({baseline_acc:.1f}%)')
ax.set_xlabel('sparsity (%)')
ax.set_ylabel('test accuracy (%)')
ax.set_title('Q7 — Pruning Before vs. After Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q7_prune_before_vs_after.png', dpi=150)
plt.show()

print(f'\n{"sparsity":>10}  {"post-train":>12}  {"at-init":>10}  {"diff":>10}')
print('-' * 48)
for sp, pt, ai in zip(sparsity_levels, acc_after_ft, acc_prune_at_init):
    print(f'{sp:>10.0%}  {pt:>12.2f}%  {ai:>10.2f}%  {pt-ai:>+9.2f}%')

### Q7 Reflection

**Post-training pruning vs pruning at init:**  
The main difference is what information the pruning decision is based on. When I prune after training, magnitude is actually meaningful — a weight that stayed small through thousands of gradient steps across the whole dataset is probably not doing much. Pruning at init is basically just randomly zeroing weights since random initialization values say nothing about importance.

**Why post-training wins at high sparsity:**  
At 25% sparsity the two approaches are pretty close — the network has enough redundancy that it doesn't matter much which weights you remove. At 90% sparsity the gap opens up significantly. Post-training pruning has kept the weights the network actually relied on, so fine-tuning has a reasonable starting point. Pruning at init might have zeroed out connections that would've been critical, and now training is stuck trying to compensate with a fixed sparse structure that wasn't designed for the task.

This connects to the lottery ticket hypothesis — the idea that within a dense network there are sparse subnetworks that could train well if you found them. Finding one at random at 90% sparsity is pretty unlikely, whereas post-training magnitude pruning is a rough approximation of finding a good one.

---
## Q8: GPTQ — Written Answers

Full answers are in the PDF writeup. Quick summary for reference:

1. **Main innovations:** GPTQ adapts the Optimal Brain Quantization (OBQ) framework — it quantizes one weight at a time and adjusts the remaining weights in the same row to compensate for the rounding error. The Cholesky reformulation of the Hessian inverse is what makes this feasible at 175B scale (one factorization upfront instead of a full $O(d^3)$ inversion at every step).

2. **Scalability:** Lazy batch updates (accumulating corrections for 128 columns at a time rather than one) reduce memory traffic dramatically. The Hessian is estimated from ~128 calibration samples, which avoids backprop entirely — just a single forward pass is enough to get useful second-order information.

3. **Limitation:** GPTQ only quantizes weights, not activations. On hardware with INT8 tensor cores (like Ampere/Hopper GPUs), you still have to dequantize back to FP16 before matrix multiplies, so the speedup is mostly from reduced memory bandwidth, not compute. One direction to fix this is combining GPTQ with something like SmoothQuant, which can bring activations into a range that's easier to quantize without blowing up accuracy.